# pcefg: Point-Charge (PC) Model for the Electric Field Gradient (EFG)

`pcefg` is a Python package for computing the Electric Field Gradient (EFG) tensor, asymmetry parameters ($\eta$), and quadrupolar coupling constants ($\chi_Q$) in crystal structures using a classical point-charge model. It serves as a lightweight, fast alternative or complementary approach to First-Principles/Density Functional Theory (DFT) calculations.

---

## Features

- **Fast EFG Tensor Calculation**: Computes lattice EFG tensors via direct lattice summation.
- **Sternheimer Antishielding**: Supports polarization corrections via $(1-\gamma_\infty)$.
- **ASE Integration**: Works directly with Atomic Simulation Environment (`ase.Atoms`) structures.
- **Crystalline Symmetry Support**: Automatically handles spacegroup site labels and site-specific charge specifications.
- **Quadrupole Coupling Utilities**: Calculates $V_{zz}$, $\eta$, and quadrupolar coupling constants ($\chi_Q$) for arbitrary spin $I > 1/2$ nuclei.

---

## Theoretical Background

### Model Hamiltonian

The nuclear quadrupole interaction Hamiltonian ($\hat{\mathcal{H}}_Q$) describes the coupling between a non-spherical nucleus
(with spin $I > \frac{1}{2}$ and electric quadrupole moment $Q$) and the local electric field gradient (EFG) generated by its surrounding
electronic environment:

$$\hat{\mathcal{H}}_Q = \sum_{i}^{N_{\mathrm{nuc}}}\frac{eQ^i(1-\gamma_\infty^i)}{\hbar\,2I(2I-1)} \sum_{\alpha\beta} V_{\alpha\beta}^{i} \hat{I}_\alpha^i \hat{I}_\beta^i, \quad \alpha, \beta = x, y, z$$

where:

- $Q^i$ is the $i$-th nuclear electric quadrupole moment.
- $V_{\alpha\beta}^{i}$ is the EFG tensor at the site of the $i$-th quadrupolar nucleus.
- $\hat{I}_{\alpha}^i$ and $\hat{I}_{\beta}^i$ are the nuclear spin operators.
- $\gamma_\infty^i$ is the Sternheimer antishielding factor.

Here $V_{\alpha\beta}^i \equiv \partial_{\alpha}\partial_{\beta} V(\mathbf{r}_i)$, with $V(\mathbf{r}_i)$ being the electrostatic potential evaluated at the nucleus. Notice that since the electric field is $\mathbf{E} = -\nabla V(\mathbf{r})$, the EFG tensor can also be expressed as $V_{\alpha\beta} = -\partial_{\alpha}E_{\beta}$.

To obtain the EFG, we start from the electrostatic potential centered at the nuclear site $V(\mathbf{r}_0)$:

$$V(\mathbf{r}_0)=\frac{1}{4\pi\varepsilon_0} \int d\mathbf{r}' \frac{\rho(\mathbf{r}')}{\lvert\mathbf{r}_0-\mathbf{r}'\rvert}$$

### Point-Charge EFG Model

In an ionic crystal, the EFG at a particular site depends on the charge distribution of the surrounding ions. The simplest model treats the ions as stationary point charges located at lattice sites.

Assuming a collection of point charges $\rho(\mathbf{r})=\sum_k q_k\delta(\mathbf{r}-\mathbf{r}_k)$, the potential simplifies to:

$$V(\mathbf{r}_0)= \frac{1}{4\pi\varepsilon_0} \sum_k \frac{q_k}{x_k}$$

$$\implies \quad \delta(\mathbf{r}'-\mathbf{r}_k) \ne 0 \quad \text{iff} \quad \mathbf{r}'=\mathbf{r}_k$$

where $q_k$ and $\mathbf{x}_k = \mathbf{r}_0 - \mathbf{r}_k$ are the charge and position vector of the $k$-th ion (with coordinates $\mathbf{x}_k=(x_{1k}, x_{2k}, x_{3k})$) located at distance $r_k = \lvert\mathbf{x}_k\rvert$ from the origin at the site of interest ($\mathbf{r}_0$).

The EFG tensor components $V_{ij} = \partial^2 V / \partial x_i\partial x_j$ due to this periodic array of point charges are given by:

$$V_{ij} = \frac{1}{4\pi\varepsilon_0} \sum_k q_k \left( \frac{3x_{ik}x_{jk}-\delta_{ij}r_k^2}{r_k^5} \right), \quad i, j = 1, 2, 3$$

where $\delta_{ij}$ is the Kronecker delta.


### Sternheimer Antishielding Correction

To account for the polarization of the core electronic cloud surrounding the probe nucleus, the lattice EFG is scaled using the Sternheimer antishielding factor $\gamma_\infty$:

$$V_{ij}^{\mathrm{total}} = (1-\gamma_\infty)\, V_{ij}^{\mathrm{lattice}}$$


#### Calculated Properties

The diagonalization of the EFG tensor yields its principal components $(V_{xx}, V_{yy}, V_{zz})$, ordered by magnitude:

$$\vert{}V_{zz}\vert{} \ge \vert{}V_{yy}\vert{} \ge \vert{}V_{xx}\vert{}$$

From these components, the asymmetry parameter $\eta$ and quadrupolar coupling constant $\chi_Q$ are calculated:

$$\eta = \frac{V_{xx} - V_{yy}}{V_{zz}}, \qquad \chi_Q = \frac{e Q V_{zz}}{h}$$

---


<!-- 
The electrostatic potential at a probe position $\mathbf{r}_0$ is:

$$V(\mathbf{r}_0)=\frac{1}{4\pi\varepsilon_0} \int \frac{\rho(\mathbf{r}')}{\vert{}\mathbf{r}'-\mathbf{r}_0\vert{}}\,d\tau'$$

Assuming a collection of point charges $\rho(\mathbf{r})=\sum_k q_k\delta(\mathbf{r}-\mathbf{r}_k)$, the potential simplifies to:

$$V(\mathbf{r}_0)= \frac{1}{4\pi\varepsilon_0} \sum_k \frac{q_k}{R_k}$$

where $\mathbf{R}_k = \mathbf{r}_0 - \mathbf{r}_k$ and $R_k = \vert{}\mathbf{R}_k\vert{}$.

The EFG tensor is defined as the Hessian of the electrostatic potential:

$$V_{ij} = \frac{\partial^2 V}{\partial x_i\partial x_j}$$

Evaluating the partial derivatives yields the explicit sum over point charges:

$$V_{ij} = \frac{1}{4\pi\varepsilon_0} \sum_k q_k \left( \frac{3R_{k,i}R_{k,j}-\delta_{ij}R_k^2}{R_k^5} \right)$$

where $\delta_{ij}$ is the Kronecker delta.

### Sternheimer Antishielding Correction

To account for the polarization of the core electronic cloud surrounding the probe nucleus, the lattice EFG is scaled using the Sternheimer antishielding factor $\gamma_\infty$:

$$V_{ij}^{\mathrm{total}} = (1-\gamma_\infty)\, V_{ij}^{\mathrm{lattice}}$$

### Quadrupolar Interaction

For nuclei with spin $I > \frac{1}{2}$, the electric quadrupole interaction contribution to the Hamiltonian is:

$$\hat{\mathcal{H}}_Q = \sum_{i}^{N_{\mathrm{nuc}}}\frac{eQ^i(1-\gamma_\infty^i)}{\hbar\,2I(2I-1)} \sum_{\alpha\beta} V_{\alpha\beta}^{i} \hat{I}_\alpha^i \hat{I}_\beta^i$$

where:
- $Q^i$ is the $i$-th nuclear electric quadrupole moment.
- $V_{\alpha\beta}^{i}$ is the external EFG tensor at the site of the $i$-th quadrupolar nucleus.
- $\hat{I}_\alpha^i$ are the nuclear spin operators.

Diagonalization of the EFG tensor yields its principal components $(V_{xx}, V_{yy}, V_{zz})$, ordered by magnitude:

$$\vert{}V_{zz}\vert{} \ge \vert{}V_{yy}\vert{} \ge \vert{}V_{xx}\vert{}$$

From these components, the asymmetry parameter $\eta$ and quadrupolar coupling constant $\chi_Q$ are calculated:

$$\eta = \frac{V_{xx} - V_{yy}}{V_{zz}}, \qquad \chi_Q = \frac{e Q V_{zz}}{h}$$

---
-->